# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as a Python dict
metadata = dataset.metadata.to_json()
print("Dataset name:", metadata['name'])
print("Description:", metadata['description'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we programmatically identify record sets, their associated fields, and their `@id` identifiers. All referencing is by `@id`.

**Note:** If you want to see the entire metadata structure, uncomment the print below.

In [ ]:
# To explore the possible record sets in the dataset schema
from pprint import pprint

record_sets = []

# The Croissant schema typically describes record sets under the 'recordSet' key in metadata.
if 'recordSet' in metadata and metadata['recordSet']:
    for rs in metadata['recordSet']:
        rs_id = rs.get('@id', None)
        rs_name = rs.get('name', None)
        print(f"RecordSet @id: {rs_id} | Name: {rs_name}")
        # List fields for each record set
        fields = rs.get('field', [])
        print("  Fields:")
        for field in fields:
            field_id = None
            field_name = None
            if isinstance(field, dict):
                field_id = field.get('@id', None)
                field_name = field.get('name', None)
            elif isinstance(field, str):
                field_id = field
            print(f"    Field @id: {field_id} | Name: {field_name}")
        record_sets.append(rs_id)
else:
    print("No record sets found in the dataset metadata. Please inspect the dataset documentation or raw metadata.")

# To inspect the full structure if needed:
# pprint(metadata)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** You can replace the chosen `@id` in the cell below with another record set to analyze different parts of the dataset.

In [ ]:
# If record sets were found, extract data from them
dataframes = {}
if record_sets:
    for record_set_id in record_sets:
        print(f"Loading records from RecordSet @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    # Choose the first available record set for demonstration
    selected_record_set = record_sets[0]
    print(f"\nColumns in DataFrame for RecordSet {selected_record_set}:")
    print(dataframes[selected_record_set].columns.tolist())
    display(dataframes[selected_record_set].head())
else:
    print("No record sets to extract data from.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we will:
- Select a numeric field for analysis (update with appropriate `@id`/column).
- Filter records, normalize values, group by a categorical attribute.

**Note:** Make sure to adjust the `numeric_field_id` and `group_field_id` to reference available columns (by their Croissant `@id`) from your RecordSet.

In [ ]:
# Example EDA
# ---- Configure these variables based on your data structure. ----
# For demonstration, we auto-select the first numeric column.

import numpy as np

if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    numeric_field_id = None
    group_field_id = None
    # Auto-detect one numeric field for demonstration
    for col in df.columns:
        # Try to convert the column to numeric, ignore errors
        try:
            if pd.api.types.is_numeric_dtype(df[col]) or pd.to_numeric(df[col], errors='coerce').notna().sum() > 0:
                numeric_field_id = col
                break
        except:
            continue
    print(f"Selected numeric field for analysis: {numeric_field_id}")
    # Try to select a group (categorical) field
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < 15:
            group_field_id = col
            break
    if group_field_id:
        print(f"Will group data by: {group_field_id}")

    # Ensure column is numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Use mean for threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (
        filtered_df[numeric_field_id].std() + 1e-8
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by category, if possible
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No data available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Example: Histogram of the numeric field, boxplot grouped by the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if record_sets and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()


## 6. Conclusion
This notebook demonstrated how to load, explore, and process a FAIR dataset described with a Croissant schema using the `mlcroissant` library. The main steps included identifying record sets and fields by their `@id`, extracting and analyzing tabular data, performing basic normalization and grouping, and visualizing data distributions. 

For more in-depth domain analysis, refer to the dataset's README and documentation for semantic meaning of each field.